# 10 · Self-RAG + Adaptive RAG with LangGraph
**Adaptive (route first):** classify complexity → answer directly / retrieve once / decompose and retrieve iteratively.
**Self-RAG (reflect after):** is the answer grounded? does it answer the question? If not, rewrite and retry, with a retry limit.
*Our approximation uses LLM graders; the original Self-RAG trains reflection tokens and Adaptive-RAG trains a small classifier.*

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from rag_common import format_docs, doc_ids

MAX_RETRIES = 2
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

class State(TypedDict, total=False):
    question: str
    query: str
    complexity: str
    documents: list[Document]
    answer: str
    retries: int
    path: list[str]

class Complexity(BaseModel):
    level: Literal["simple", "single_hop", "multi_hop"] = Field(
        description="simple: general knowledge, no documents needed; single_hop: one lookup; multi_hop: needs several lookups")

class SubQuestions(BaseModel):
    questions: list[str] = Field(description="2-3 standalone sub-questions")

class YesNo(BaseModel):
    verdict: Literal["yes", "no"]

classifier = model.with_structured_output(Complexity)
decomposer = model.with_structured_output(SubQuestions)
judge = model.with_structured_output(YesNo)

## Nodes

In [ ]:
def classify(s: State) -> State:
    level = classifier.invoke(f"Classify this TAC question:\n{s['question']}").level
    print(f"   🧭 complexity: {level}")
    return {"complexity": level, "query": s["question"], "retries": 0, "path": [f"classify:{level}"]}

def answer_direct(s: State) -> State:
    return {"answer": model.invoke(s["question"]).content, "path": s["path"] + ["answer_direct"]}

def retrieve_once(s: State) -> State:
    return {"documents": retriever.invoke(s["query"]), "path": s["path"] + ["retrieve_once"]}

def retrieve_iterative(s: State) -> State:
    subs = decomposer.invoke(f"Break this into sub-questions:\n{s['question']}").questions
    docs = []
    for sq in subs:
        print(f"   🔁 sub-question: {sq}")
        docs += retriever.invoke(sq)
    unique = list({d.metadata["chunk_id"]: d for d in docs}.values())
    return {"documents": unique, "path": s["path"] + ["retrieve_iterative"]}

def generate(s: State) -> State:
    ans = model.invoke(f"Answer ONLY from the context, citing [doc_id].\n<context>\n{format_docs(s['documents'])}\n</context>\n"
                       f"Question: {s['question']}").content
    return {"answer": ans, "path": s["path"] + ["generate"]}

def rewrite(s: State) -> State:
    q = model.invoke(f"Rewrite as a better search query. Reply with the query only.\n{s['question']}").content
    return {"query": q, "retries": s["retries"] + 1, "path": s["path"] + ["rewrite"]}

## Reflection edges (Self-RAG)

In [ ]:
def route(s: State) -> str:
    return {"simple": "answer_direct", "single_hop": "retrieve_once", "multi_hop": "retrieve_iterative"}[s["complexity"]]

def reflect(s: State) -> str:
    if s["retries"] >= MAX_RETRIES:
        print("   ⛔ retry limit reached: returning best effort")
        return "end"
    grounded = judge.invoke(f"Is every claim in the ANSWER supported by the DOCUMENTS?\nDOCUMENTS:\n{format_docs(s['documents'])}\nANSWER:\n{s['answer']}").verdict
    if grounded == "no":
        print("   🪞 not grounded → regenerate")
        return "retry_generate"
    useful = judge.invoke(f"Does the ANSWER resolve the QUESTION?\nQUESTION: {s['question']}\nANSWER: {s['answer']}").verdict
    if useful == "no":
        print("   🪞 grounded but not useful → rewrite query")
        return "rewrite"
    print("   🪞 grounded and useful ✓")
    return "end"

def bump(s: State) -> State:                # count regeneration attempts too
    return {"retries": s["retries"] + 1, "path": s["path"] + ["retry"]}

g = StateGraph(State)
for name, fn in [("classify", classify), ("answer_direct", answer_direct), ("retrieve_once", retrieve_once),
                 ("retrieve_iterative", retrieve_iterative), ("generate", generate), ("rewrite", rewrite), ("retry_generate", bump)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route, ["answer_direct", "retrieve_once", "retrieve_iterative"])
g.add_edge("answer_direct", END)
g.add_edge("retrieve_once", "generate")
g.add_edge("retrieve_iterative", "generate")
g.add_conditional_edges("generate", reflect, {"end": END, "retry_generate": "retry_generate", "rewrite": "rewrite"})
g.add_edge("retry_generate", "generate")
g.add_edge("rewrite", "retrieve_once")
app = g.compile()
print(app.get_graph().draw_mermaid())

## Run it on three levels of difficulty

In [ ]:
for q in ["What does the acronym BFD stand for?",
          "Clients drop when roaming between floors on the Orion controller. What's the fix?",
          "Which Aurora AR-900 release fixes the strict-priority queue starving best-effort traffic, and what should a 9.2.1 customer do until they upgrade?"]:
    print(f"\n❓ {q}")
    out = app.invoke({"question": q})
    print("   path:", " → ".join(out["path"]))
    print("💬", out["answer"][:600])

---
**🔧 TAC takeaway:** Triage effort to the ticket (no war room for a P4) and peer-review the answer before it goes out. Retry limits make sure a hard question ends in an honest answer, not an endless loop.